In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install gradio==4.12 transformers sentencepiece
!apt-get install ffmpeg -y


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-74oe1o_n
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-74oe1o_n
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
import gradio as gr
import whisper
from transformers import pipeline
import tempfile
import os

# Load models
whisper_model = whisper.load_model("small")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def process_audio(audio_path):
    if audio_path is None:
        return "No audio uploaded", ""

    # Convert to a safe temporary WAV
    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav").name
    os.system(f"ffmpeg -y -i '{audio_path}' -ar 16000 -ac 1 '{temp_wav}'")

    # Whisper transcription (Python API — RELIABLE)
    result = whisper_model.transcribe(temp_wav, fp16=False, language="en")
    transcript = result["text"]

    # Summarize the text
    summary = summarizer(transcript, max_length=200, min_length=60)[0]["summary_text"]

    return transcript, summary


ui = gr.Interface(
    fn=process_audio,
    inputs=gr.Audio(type="filepath", label="🎤 Upload Audio (MP3/WAV)"),
    outputs=[
        gr.Textbox(label="📄 Transcript", lines=10),
        gr.Textbox(label="📝 Summary", lines=10),
    ],
    title="🎙️ Milestone 4 – Meeting Summarizer (Stable Version)",
    description="Upload audio, Whisper transcribes it, BART summarizes it."
)

ui.launch(debug=True)


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 4.12.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://fc337e7f6216f70d39.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Your max_length is set to 200, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)
Your max_length is set to 200, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)
